# 60_01 - CustomViT v1 From Scratch

This notebook trains and evaluates the `customvit_v1` Vision Transformer from scratch using the shared rerun-safe training pipeline.


In [1]:
import json
import os
import random
import sys
from pathlib import Path

import mlflow
import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOTEBOOK_DIR = Path.cwd().expanduser().resolve()


def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "requirements.txt"]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers_all) and any((candidate / marker).exists() for marker in markers_any):
            return candidate
    raise RuntimeError(f"Could not locate project root from: {start}")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
DATA_DIR = PROJECT_ROOT / "data"
CONFIGS_DIR = PROJECT_ROOT / "configs"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
TRACKING_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
SPLIT_DIR = DATA_DIR / "splits" / "split_v1"
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
TEST_CSV = SPLIT_DIR / "test.csv"
CLASSES_JSON = SPLIT_DIR / "classes.json"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset_loader import ImageDataset
from src.data.transforms import apply_size_overrides, get_eval_transforms, get_train_transforms, load_transforms_config
from src.models.vit_scratch import (
    TrainingHistory,
    attempt_onnx_export,
    atomic_save_json,
    benchmark_inference,
    build_experiment_signature,
    build_metrics_payload,
    build_model,
    build_training_config,
    collect_device_info,
    configure_trainable_stage,
    count_parameters,
    create_grad_scaler,
    ensure_dir,
    evaluate_model,
    fit_model,
    get_model_spec,
    get_recommended_image_size,
    get_recommended_resize_size,
    get_trainable_parameter_groups,
    load_training_resume_state,
    model_size_mb_from_state_dict,
    resolve_run_directory,
    restore_best_weights,
    save_checkpoint_atomic,
    save_report_metrics_copy,
    save_training_curves,
)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server


In [3]:
MODEL_NAME = "customvit_v1"
NOTEBOOK_NAME = "60_01_customvit_v1.ipynb"
MODEL_DESCRIPTION = "Educational encoder-only Vision Transformer from scratch."
BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 0.0003
WEIGHT_DECAY = 0.0001
HEAD_LR = LEARNING_RATE
BACKBONE_LR = LEARNING_RATE
GRAD_CLIP_MAX_NORM = 1.0
LR_FACTOR = 0.3
LR_PATIENCE = 3
SEED = 42
SPLIT_ID = "split_v1"
DATASET_VERSION = "prepared_current"

INPUT_IMAGE_SIZE = get_recommended_image_size(MODEL_NAME)
EVAL_RESIZE_SIZE = get_recommended_resize_size(MODEL_NAME)
TRANSFORM_ID_TRAIN = f"transforms_v1_train_runtime_aug_crop{INPUT_IMAGE_SIZE}"
TRANSFORM_ID_EVAL = f"transforms_v1_eval_resize{EVAL_RESIZE_SIZE}_centercrop{INPUT_IMAGE_SIZE}_imagenetnorm"

MODEL_FAMILY_DIR = MODELS_DIR / "vit_scratch" / MODEL_NAME
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FAMILY_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

required_paths = {
    "train_csv": TRAIN_CSV,
    "val_csv": VAL_CSV,
    "test_csv": TEST_CSV,
    "classes_json": CLASSES_JSON,
    "transforms_config": TRANSFORMS_CONFIG_PATH,
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs for {MODEL_NAME}: {missing}")

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    classes_to_idx = json.load(f)
NUM_CLASSES = len(classes_to_idx)

base_cfg = load_transforms_config(TRANSFORMS_CONFIG_PATH)
cfg = apply_size_overrides(base_cfg, image_size=INPUT_IMAGE_SIZE, resize_size=EVAL_RESIZE_SIZE)
train_tfm = get_train_transforms(cfg)
eval_tfm = get_eval_transforms(cfg)

train_ds = ImageDataset(
    split_csv=TRAIN_CSV,
    transform=train_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)
val_ds = ImageDataset(
    split_csv=VAL_CSV,
    transform=eval_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)
test_ds = ImageDataset(
    split_csv=TEST_CSV,
    transform=eval_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = DEVICE == "cuda"
NUM_WORKERS = min(8, os.cpu_count() or 2)
PIN_MEMORY = DEVICE == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

device_info = collect_device_info(
    device=DEVICE,
    amp_enabled=AMP_ENABLED,
    num_workers=NUM_WORKERS,
    batch_size=BATCH_SIZE,
)
spec = get_model_spec(MODEL_NAME)

training_params = {
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
    "lr_factor": LR_FACTOR,
    "lr_patience": LR_PATIENCE,
    "seed": SEED,
    "amp_enabled": AMP_ENABLED,
    "input_image_size": INPUT_IMAGE_SIZE,
    "eval_resize_size": EVAL_RESIZE_SIZE,
}

signature_payload = {
    "model_name": MODEL_NAME,
    "split_id": SPLIT_ID,
    "transform_id_train": TRANSFORM_ID_TRAIN,
    "transform_id_eval": TRANSFORM_ID_EVAL,
    "training_params": training_params,
    "architecture": spec.to_dict(),
}
experiment_signature = build_experiment_signature(signature_payload)
resolution = resolve_run_directory(MODEL_FAMILY_DIR, experiment_signature, allow_resume=True)
RUN_ACTION = resolution.action
RUN_DIR = resolution.run_dir

ensure_dir(RUN_DIR)
CHECKPOINT_PATH = RUN_DIR / "checkpoint.pt"
CONFIG_PATH = RUN_DIR / "config.json"
METRICS_PATH = RUN_DIR / "metrics.json"
ONNX_PATH = RUN_DIR / "exported.onnx"

config = build_training_config(
    model_name=MODEL_NAME,
    backbone_name=spec.family,
    weights_name="scratch",
    split_id=SPLIT_ID,
    transform_id_train=TRANSFORM_ID_TRAIN,
    transform_id_eval=TRANSFORM_ID_EVAL,
    dataset_version=DATASET_VERSION,
    training_params=training_params,
    experiment_signature=experiment_signature,
    extra={
        "device_info": device_info,
        "notebook_name": NOTEBOOK_NAME,
        "model_description": MODEL_DESCRIPTION,
        "run_action": RUN_ACTION,
        "input_image_size": INPUT_IMAGE_SIZE,
        "eval_resize_size": EVAL_RESIZE_SIZE,
        "architecture": spec.to_dict(),
        "model_family_group": "vit_scratch",
    },
)

SKIP_TRAINING = RUN_ACTION == "reuse" and CONFIG_PATH.exists() and METRICS_PATH.exists() and CHECKPOINT_PATH.exists()

print("MODEL_NAME          :", MODEL_NAME)
print("RUN_ACTION          :", RUN_ACTION)
print("RUN_DIR             :", RUN_DIR)
print("EXPERIMENT_SIGNATURE:", experiment_signature)
print("DEVICE              :", DEVICE)
print("INPUT_IMAGE_SIZE    :", INPUT_IMAGE_SIZE)
print("EVAL_RESIZE_SIZE    :", EVAL_RESIZE_SIZE)
print("ARCHITECTURE        :", spec.to_dict())


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


MODEL_NAME          : customvit_v1
RUN_ACTION          : create
RUN_DIR             : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/vit_scratch/customvit_v1/run_20260505_102351
EXPERIMENT_SIGNATURE: 13e6c334adcd908a
DEVICE              : cuda
INPUT_IMAGE_SIZE    : 224
EVAL_RESIZE_SIZE    : 256
ARCHITECTURE        : {'name': 'customvit_v1', 'family': 'vit_scratch', 'image_size': 224, 'resize_size': 256, 'patch_size': 16, 'embed_dim': 192, 'depth': 6, 'num_heads': 3, 'mlp_ratio': 4.0, 'dropout_p': 0.1, 'attention_dropout_p': 0.0, 'num_classes': 3, 'input_channels': 3}


In [4]:
if SKIP_TRAINING:
    existing_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    existing_metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    print("[SKIP] Matching completed run already exists:", RUN_DIR)
    print("Existing best epoch:", existing_metrics.get("best_epoch"))
    print("Existing test accuracy:", existing_metrics.get("test_accuracy"))
    FINAL_CONFIG = existing_config
    FINAL_METRICS = existing_metrics
else:
    atomic_save_json(CONFIG_PATH, config)

    model = build_model(
        model_name=MODEL_NAME,
        num_classes=NUM_CLASSES,
        input_channels=3,
    ).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    configure_trainable_stage(model, MODEL_NAME, "full_train")
    param_groups = get_trainable_parameter_groups(
        model=model,
        model_name=MODEL_NAME,
        head_lr=HEAD_LR,
        backbone_lr=BACKBONE_LR,
        weight_decay=WEIGHT_DECAY,
    )
    optimizer = torch.optim.AdamW(param_groups)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE)
    scaler = create_grad_scaler(DEVICE, AMP_ENABLED)

    history = TrainingHistory()
    best_state = {}
    start_epoch = 0

    if RUN_ACTION == "resume" and CHECKPOINT_PATH.exists():
        resume_state = load_training_resume_state(
            checkpoint_path=CHECKPOINT_PATH,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            experiment_signature=experiment_signature,
            map_location="cpu",
        )
        history = resume_state.history
        best_state = resume_state.best_state
        start_epoch = resume_state.start_epoch
        print("[RESUME] Completed epochs already recorded:", start_epoch)

    history, best_state = fit_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        num_classes=NUM_CLASSES,
        epochs=EPOCHS,
        model_name=MODEL_NAME,
        checkpoint_path=CHECKPOINT_PATH,
        config=config,
        experiment_signature=experiment_signature,
        history=history,
        best_state=best_state,
        start_epoch=start_epoch,
        amp_enabled=AMP_ENABLED,
        scaler=scaler,
        grad_clip_max_norm=GRAD_CLIP_MAX_NORM,
    )

    model = restore_best_weights(model, best_state)
    test_metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE,
        num_classes=NUM_CLASSES,
        amp_enabled=AMP_ENABLED,
    )
    benchmark_metrics = benchmark_inference(
        model=model,
        loader=test_loader,
        device=DEVICE,
        warmup_batches=5,
        timed_batches=20,
        amp_enabled=AMP_ENABLED,
    )
    curve_paths = save_training_curves(history, RUN_DIR)

    onnx_export_status = attempt_onnx_export(
        model=model,
        export_path=ONNX_PATH,
        input_shape=(1, 3, INPUT_IMAGE_SIZE, INPUT_IMAGE_SIZE),
        device=DEVICE,
    )
    if not onnx_export_status["succeeded"]:
        print("[WARNING] ONNX export failed:", onnx_export_status["error"])

    metrics_payload = build_metrics_payload(
        history=history,
        best_state=best_state,
        test_metrics=test_metrics,
        benchmark_metrics=benchmark_metrics,
        parameter_count=count_parameters(model),
        trainable_parameter_count=count_parameters(model, trainable_only=True),
        model_size_mb=model_size_mb_from_state_dict(model),
        device_info=device_info,
        onnx_export_status=onnx_export_status,
    )
    atomic_save_json(METRICS_PATH, metrics_payload)
    report_copy_path = save_report_metrics_copy(REPORTS_METRICS_DIR, MODEL_NAME, RUN_DIR.name, metrics_payload)

    save_checkpoint_atomic(
        CHECKPOINT_PATH,
        {
            "model_name": MODEL_NAME,
            "experiment_signature": experiment_signature,
            "completed_epoch": int(EPOCHS),
            "config": config,
            "history": history.to_dict(),
            "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "best_state": best_state,
        },
    )

    with mlflow.start_run(run_name=f"{MODEL_NAME}_{RUN_DIR.name}"):
        mlflow.log_param("stage", "vit_scratch_training")
        mlflow.log_param("model_name", MODEL_NAME)
        mlflow.log_param("backbone_name", spec.family)
        mlflow.log_param("weights_name", "scratch")
        mlflow.log_param("split_id", SPLIT_ID)
        mlflow.log_param("transform_id_train", TRANSFORM_ID_TRAIN)
        mlflow.log_param("transform_id_eval", TRANSFORM_ID_EVAL)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("learning_rate", LEARNING_RATE)
        mlflow.log_param("weight_decay", WEIGHT_DECAY)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("device", DEVICE)
        mlflow.log_param("amp_enabled", AMP_ENABLED)
        mlflow.log_param("input_image_size", INPUT_IMAGE_SIZE)
        mlflow.log_param("eval_resize_size", EVAL_RESIZE_SIZE)
        mlflow.log_param("patch_size", spec.patch_size)
        mlflow.log_param("embed_dim", spec.embed_dim)
        mlflow.log_param("depth", spec.depth)
        mlflow.log_param("num_heads", spec.num_heads)
        mlflow.log_param("mlp_ratio", spec.mlp_ratio)
        mlflow.log_param("experiment_signature", experiment_signature)
        mlflow.log_param("run_action", RUN_ACTION)

        mlflow.log_metric("best_val_macro_f1", float(best_state.get("best_val_macro_f1", float("nan"))))
        mlflow.log_metric("best_val_loss", float(best_state.get("best_val_loss", float("nan"))))
        mlflow.log_metric("best_val_accuracy", float(best_state.get("best_val_accuracy", float("nan"))))
        mlflow.log_metric("test_loss", float(test_metrics["loss"]))
        mlflow.log_metric("test_accuracy", float(test_metrics["accuracy"]))
        mlflow.log_metric("test_macro_f1", float(test_metrics["macro_f1"]))
        mlflow.log_metric("latency_ms_per_batch", float(benchmark_metrics["latency_ms_per_batch"]))
        mlflow.log_metric("latency_ms_per_image", float(benchmark_metrics["latency_ms_per_image"]))
        mlflow.log_metric("throughput_img_per_sec", float(benchmark_metrics["throughput_img_per_sec"]))
        mlflow.log_metric("parameter_count", float(count_parameters(model)))
        mlflow.log_metric("trainable_parameter_count", float(count_parameters(model, trainable_only=True)))
        mlflow.log_metric("model_size_mb", float(model_size_mb_from_state_dict(model)))

        mlflow.log_artifact(str(CONFIG_PATH))
        mlflow.log_artifact(str(METRICS_PATH))
        mlflow.log_artifact(str(CHECKPOINT_PATH))
        mlflow.log_artifact(str(curve_paths["loss_curve"]))
        mlflow.log_artifact(str(curve_paths["accuracy_curve"]))
        if ONNX_PATH.exists():
            mlflow.log_artifact(str(ONNX_PATH))

    print("[OK] Metrics saved:", METRICS_PATH)
    print("[OK] Report metrics copy saved:", report_copy_path)
    print("[OK] Best epoch:", best_state.get("epoch"))
    print("[OK] Test accuracy:", metrics_payload.get("test_accuracy"))

    FINAL_CONFIG = config
    FINAL_METRICS = metrics_payload

print(json.dumps({
    "model_name": MODEL_NAME,
    "run_action": RUN_ACTION,
    "run_dir": str(RUN_DIR),
    "experiment_signature": experiment_signature,
    "best_epoch": FINAL_METRICS.get("best_epoch"),
    "test_accuracy": FINAL_METRICS.get("test_accuracy"),
    "test_macro_f1": FINAL_METRICS.get("test_macro_f1"),
}, indent=2))


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[full_train] [Epoch 01/40] train_loss=0.9992 train_acc=0.4675 val_loss=0.9387 val_acc=0.5177 val_f1=0.4940 lr_backbone=0.000300 lr_head=0.000300


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[full_train] [Epoch 02/40] train_loss=0.8613 train_acc=0.5827 val_loss=0.7510 val_acc=0.6631 val_f1=0.6657 lr_backbone=0.000300 lr_head=0.000300


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


...

[full_train] [Epoch 08/40] train_loss=0.5382 train_acc=0.7654 val_loss=0.5007 val_acc=0.7869 val_f1=0.7957 lr_backbone=0.000300 lr_head=0.000300
[full_train] [Epoch 09/40] train_loss=0.5148 train_acc=0.7763 val_loss=0.4861 val_acc=0.7933 val_f1=0.8004 lr_backbone=0.000300 lr_head=0.000300


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[full_train] [Epoch 10/40] train_loss=0.4847 train_acc=0.7929 val_loss=0.4283 val_acc=0.8238 val_f1=0.8314 lr_backbone=0.000300 lr_head=0.000300
[full_train] [Epoch 11/40] train_loss=0.4664 train_acc=0.8010 val_loss=0.4016 val_acc=0.8350 val_f1=0.8417 lr_backbone=0.000300 lr_head=0.000300
[full_train] [Epoch 12/40] train_loss=0.4487 train_acc=0.8095 val_loss=0.3919 val_acc=0.8388 val_f1=0.8445 lr_backbone=0.000300 lr_head=0.000300
[full_train] [Epoch 13/40] train_loss=0.4322 train_acc=0.8166 val_loss=0.3620 val_acc=0.8514 val_f1=0.8569 lr_backbone=0.000300 lr_head=0.000300
[full_train] [Epoch 14/40] train_loss=0.4110 train_acc=0.8274 val_loss=0.3413 val_acc=0.8610 val_f1=0.8666 lr_backbone=0.000300 lr_head=0.000300
[full_train] [Epoch 15/40] train_loss=0.3952 train_acc=0.8344 val_loss=0.3979 val_acc=0.8390 val_f1=0.8423 lr_backbone=0.000300 lr_head=0.000300
[full_train] [Epoch 16/40] train_loss=0.3793 train_acc=0.8423 val_loss=0.3075 val_acc=0.8738 val_f1=0.8801 lr_backbone=0.000300 lr

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[full_train] [Epoch 32/40] train_loss=0.1963 train_acc=0.9194 val_loss=0.1923 val_acc=0.9242 val_f1=0.9281 lr_backbone=0.000090 lr_head=0.000090
[full_train] [Epoch 33/40] train_loss=0.1867 train_acc=0.9240 val_loss=0.1832 val_acc=0.9290 val_f1=0.9324 lr_backbone=0.000090 lr_head=0.000090
[full_train] [Epoch 34/40] train_loss=0.1841 train_acc=0.9251 val_loss=0.1686 val_acc=0.9354 val_f1=0.9390 lr_backbone=0.000090 lr_head=0.000090
[full_train] [Epoch 35/40] train_loss=0.1759 train_acc=0.9278 val_loss=0.1663 val_acc=0.9360 val_f1=0.9400 lr_backbone=0.000090 lr_head=0.000090
[full_train] [Epoch 36/40] train_loss=0.1736 train_acc=0.9297 val_loss=0.1852 val_acc=0.9315 val_f1=0.9350 lr_backbone=0.000090 lr_head=0.000090
[full_train] [Epoch 37/40] train_loss=0.1731 train_acc=0.9294 val_loss=0.1743 val_acc=0.9319 val_f1=0.9356 lr_backbone=0.000090 lr_head=0.000090
[full_train] [Epoch 38/40] train_loss=0.1676 train_acc=0.9315 val_loss=0.1708 val_acc=0.9371 val_f1=0.9405 lr_backbone=0.000090 lr